In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

def load_files(folder, filelist, delimiter=" ", skiprows=0, usecols=None):
    dataframes_dict = {}

    for file in filelist:
        full_path = os.path.join(folder, file)

        if os.path.exists(full_path):
            try:
                dataframes_dict[file] = pd.read_csv(full_path, delimiter=delimiter, skiprows=skiprows, usecols=usecols)
            except Exception as e:
                print(f"Error reading '{file}': {e}")
        else:
            print(f"Warning: File not found in '{file}'. Skipping.")
    return dataframes_dict


def spatial_moving_avg(df, window_x:int):
    df_sorted = df.sort_values(by=df.columns[1])
    x = df_sorted.iloc[:, 1].to_numpy()
    y = df_sorted.iloc[:, 0].to_numpy()

    half_w = window_x / 2.0

    left_idx = np.searchsorted(x, x - half_w, side='left')
    right_idx = np.searchsorted(x, x + half_w, side='right')

    cum_sum = np.insert(np.cumsum(y), 0, 0)
    window_sums = cum_sum[right_idx] - cum_sum[left_idx]
    counts = right_idx - left_idx
    y_ma = window_sums / counts

    return x, y_ma

def percent_diff(a: int, b: int, leva: str, levb:str):
    num = abs(a-b)
    dom = (a+b) / 2
    diff = (num/dom) * 100
    print(f"{leva}/{levb}: {diff}%")

In [ ]:
shock_speed = load_files("/home/isaac/2D34k7_6", ["shockspeed_bot.txt", "shockspeed_top.txt"])
x, y_moving_avg = spatial_moving_avg(shock_speed["shockspeed_bot.txt"], 1e7)

fig, ax = plt.subplots(1,1, figsize=(8,4))
ax.plot(shock_speed["shockspeed_bot.txt"].iloc[:,1], shock_speed["shockspeed_bot.txt"].iloc[:,0], lw=0.8, label="Speed Data")
ax.plot(x, y_moving_avg, lw=1, label = "Moving avg")
x_bot_sub = x[189:199]
y_bot_ma_sub = np.mean(y_moving_avg[189:199])
ax.hlines(y_bot_ma_sub, min(x_bot_sub), max(x_bot_sub), color="black", label=rf"$D_{{avg}}$ = {y_bot_ma_sub:e} cm/s")

#ax.set_ylim(0.3e9, 1.5e9)
#ax.set_xlim(right=0.8e9)
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, length=7)
ax.set_xlabel("x (cm)")
ax.set_ylabel(f"D (cm/s)")
ax.legend(frameon=False)
print(f"D = {y_bot_ma_sub:e} cm/s")

plt.tight_layout()

#plt.savefig("2D34_figs/2D34_Dspeed", dpi=400, transparent=True)

In [ ]:
sf = pd.read_csv('/home/isaac/2D34k7_6/sootfoil.xq', delimiter=' ', skiprows=2, usecols=[0,1,2])
sf_pivot = sf.pivot(index="Y", columns="X", values="SF")

fig, (ax1, ax2) = plt.subplots(2,1,figsize=(12,4), sharex=False, sharey=True)

for ax in (ax1, ax2):
    im = ax.imshow(
        sf_pivot.values, origin='lower', aspect='equal', cmap='plasma',
        vmin = 0.5e23,
        vmax = 2.5e23,
        extent = [sf_pivot.columns.min(),
                  sf_pivot.columns.max(),
                  sf_pivot.index.min(),
                  sf_pivot.index.max(),]
    )

ax1.set_xlim(0, 4.5e7)
ax2.set_xlim(4.5e7, 9e7)
#ax3.set_xlim(5e8, 7.5e8)
#ax4.set_xlim(7.5e8, 1e9)
ax1.set_yticks([500000])

cbar = fig.colorbar(im, ax=[ax1,ax2], location='bottom', aspect=60)

#plt.savefig("2D34_figs/2D34_sf_full", dpi=400, transparent=True)

In [ ]:
reduced_data = sf[(sf['X'] >= 7.0e7) & (sf['X'] <=8.4e7)]
pivot_df = reduced_data.pivot_table(index="Y", columns="X", values="SF")

fig, ax = plt.subplots(1, 1, figsize=(12,6))

im = ax.imshow(
    pivot_df.values,
    origin='lower',
    aspect='equal',
    cmap='plasma',
    extent=[pivot_df.columns.min(), pivot_df.columns.max(), pivot_df.index.min(), pivot_df.index.max()]
)

cbar = plt.colorbar(im, ax=ax, orientation='horizontal', aspect=50, pad=0.07, shrink=0.6)

#plt.savefig("2D34_figs/2D34_sf_partial", dpi=400, transparent=True)

In [ ]:
import numpy as np
import scipy.ndimage as ndimage

def get_cell_height(data, dx, dy, max_height, min_height=None):
    if hasattr(data, 'values'):
        data = data.values

    data = np.nan_to_num(data, nan=np.nanmean(data))
    
    Nx, Ny = data.shape

    fft_2d = np.fft.fftshift(np.fft.fft2(data)) / (Nx * Ny)
    psd_2d = np.abs(fft_2d)**2

    ky = np.fft.fftshift(np.fft.fftfreq(Ny, d=dy)) * 2 * np.pi
    psd_ky = np.sum(psd_2d, axis=0)

    min_ky = (2 * np.pi) / max_height
    max_ky = (2 * np.pi) / min_height

    valid_mask = (ky >= min_ky) & (ky <= max_ky)
    ky_pos = ky[valid_mask]
    psd_ky_pos = psd_ky[valid_mask]

    #if len(psd_ky) == 0:
        #return None, ky, psd_ky

    dominant_peak = np.argmax(psd_ky_pos)
    ky_star = ky_pos[dominant_peak]
    cell_height = (2 * np.pi) / ky_star

    return cell_height, ky_pos, psd_ky_pos

dy = np.abs(pivot_df.index[1] - pivot_df.index[0])
dx = np.abs(pivot_df.columns[1] - pivot_df.columns[0])

EXPECTED_CELL_HEIGHT = 2e7
MAX_CELL_HEIGHT = 5e7
#MIN_CELL_HEIGHT = 1e7

cell_height, ky_pos, psd_ky_pos = get_cell_height(data=pivot_df.values, dx=dx, dy=dy, max_height=MAX_CELL_HEIGHT, min_height=MIN_CELL_HEIGHT)

print(f"Cell Height: {cell_height:e} cm")

In [ ]:
import numpy as np
import scipy.ndimage as ndimage

def get_cell_length(data, dx, dy, max_height, min_height=None):
    if hasattr(data, 'values'):
        data = data.values

    data = np.nan_to_num(data, nan=np.nanmean(data))
    
    Nx, Ny = data.shape

    fft_2d = np.fft.fftshift(np.fft.fft2(data)) / (Nx * Ny)
    psd_2d = np.abs(fft_2d)**2

    kx = np.fft.fftshift(np.fft.fftfreq(Nx, d=dx)) * 2 * np.pi
    psd_kx = np.sum(psd_2d, axis=1)

    min_kx = (2 * np.pi) / max_height
    max_kx = (2 * np.pi) / min_height if min_height else np.max(kx)

    valid_mask = (kx >= min_kx) & (kx <= max_kx)
    kx_pos = kx[valid_mask]
    psd_kx_pos = psd_kx[valid_mask]

    #if len(psd_ky) == 0:
        #return None, ky, psd_ky

    dominant_peak = np.argmax(psd_kx_pos)
    kx_star = ky_pos[dominant_peak]
    cell_height = (2 * np.pi) / kx_star

    return cell_height, kx_pos, psd_kx_pos

dy = np.abs(pivot_df.index[1] - pivot_df.index[0])
dx = np.abs(pivot_df.columns[1] - pivot_df.columns[0])

EXPECTED_CELL_HEIGHT = 2e7
MAX_CELL_HEIGHT = 5e7
MIN_CELL_HEIGHT = 1e7

cell_height, kx_pos, psd_kx_pos = get_cell_length(data=pivot_df.values, dx=dx, dy=dy, max_height=MAX_CELL_HEIGHT, min_height=MIN_CELL_HEIGHT)

print(f"Cell Length: {cell_height:e} cm")